# Project 2

## Imports

In [ ]:
!pip install spacy torch textworld[gym] tqdm scikit-learn && python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 16.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.5/101.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 57.7 MB/s eta 0:00:00
  Created wheel for jericho: filename=jericho-3.3.0-py3-none-any.whl size=325237 sha256=9376bf722f694b9e53e9523aa2a4655c27c4c4ba1b44362e4a77216e79519b8b
  Stored in directory: /root/.cache/pip/wheels/a5/fa/71/4ccd66162c5fce936673106cf00a3b66b6937db5dfd0dac581
Successfully built jericho
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 145.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import sys, subprocess
print("Installing into:", sys.executable)
subprocess.check_call([sys.executable, "-m", "pip", "install", "textworld[gym]"])


In [ ]:
import os
import json
import random
import re
import numpy as np
from typing import List, Tuple, Optional
from collections import deque


import spacy
import torch
import textworld
import textworld.gym

from collections import Counter
from collections import defaultdict
from tqdm import tqdm

from spacy.tokenizer import Tokenizer
from spacy.lang.en import English
import spacy.cli
spacy.cli.download("en_core_web_sm")


from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from typing import List


## Part 1


In [ ]:
def parse_verb_object(s):
    return s.split()[0], ' '.join(s.split()[1:])

def walkthrough_and_record(game_dir, max_games=5000):

    triples = []
    seen = 0

    files = list(os.listdir(game_dir))

    # let's make it reproducible
    random.seed(42)
    random.shuffle(files)

    for file in files:

        # jsons have the walkthroughs... if no json, no need to look at this game
        if not file.endswith('.json'): continue

        # cap the total games for speed
        seen += 1
        if seen > max_games: break

        # extract walkthrough from the json meta data
        # walkthrough == winning actions
        x = json.load(open(os.path.join(game_dir, file), 'r'))
        wt = x['extras']['walkthrough']

        # setup the actual game we can interact with to get observations
        ulx = os.path.join(game_dir, file.replace('.json', '.ulx'))
        game_id = textworld.gym.register_game(ulx)
        env = textworld.gym.make(game_id)

        # record observations and track the moves we made at each to win
        obs, *_ = env.reset()

        for move in wt:
            triples.append((obs, *parse_verb_object(move)))
            obs, *_ = env.step(move)

        # don't forget the last move we just made
        triples.append((obs, *parse_verb_object(move)))

    return triples
gdir = './games/train'#local directory
print('Extracting data for ML... this could take a while')
ml_data = walkthrough_and_record(gdir)
print('> We extracted', len(ml_data), '(obs, act) pairs.')


Extracting data for ML... this could take a while


FileNotFoundError: [Errno 2] No such file or directory: 'cog2019_ftwp/games/train'

In [ ]:
def NB_model(X: List[str], y: List[str]) -> Pipeline:

    encoder = LabelEncoder()
    y = encoder.fit_transform(y)

    # split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    pipe = Pipeline([
        ("cv", CountVectorizer(ngram_range=(1,2), min_df=1, max_df=0.95)),
        ("nb", MultinomialNB())
    ])
    pipe.fit(X_train, y_train)
    y_hat = pipe.predict(X_test)
    return lambda obs: encoder.inverse_transform(pipe.predict([obs]))[0]

verb_policy = NB_model([obs for obs, *_ in ml_data], [v for _, v, _ in ml_data])


In [ ]:
nlp = English()
tokenizer = Tokenizer(nlp.vocab)

# function to create string objects out of spacy token objects
def spacy_tok(s: str):
    doc = tokenizer(str(s))
    return [t.text.lower() for t in doc]

def train_object_tagger(X, y):

    # before rest of code runs make results replicatable. Got help from chatgpt on how to do this for this model setup
    Seed = 1845
    os.environ["PYTHONHASHSEED"] = str(Seed)
    random.seed(Seed)
    np.random.seed(Seed)
    torch.manual_seed(Seed)
    torch.cuda.manual_seed_all(Seed)


    # split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # label our object classifiers outside beggining inside
    BIO2ID = {"O":0, "B":1, "I":2}
    ID2BIO = {v:k for k,v in BIO2ID.items()}

    # tokens is our X data already split into tokens  obj is the y_data
    def bio_from_object(tokens, obj):
        # normalize observation tokens to the same preprocessing used for objects
        obs = [str(t).lower().strip() for t in tokens]

        def to_list(e):
            if isinstance(e, (list, tuple)):
                return [str(x).lower().strip() for x in e if str(x).strip()]
            elif isinstance(e, str):
                return [t.lower().strip() for t in spacy_tok(str(e))]
            else:
                return []

        # Build list-of-token-lists for object spans
        if obj is None:
            phrases = []
        elif isinstance(obj, (str, list, tuple)) and not (
            isinstance(obj, (list, tuple)) and obj and isinstance(obj[0], (list, tuple))
        ):
            p = to_list(obj)
            phrases = [p] if p else []
        else:
            phrases = []
            for span in obj:
                p = to_list(span)
                if p:
                    phrases.append(p)

        # dedupe & prefer longer spans first
        seen, uniq = set(), []
        for p in phrases:
            tp = tuple(p)
            if tp and tp not in seen:
                seen.add(tp); uniq.append(p)
        phrases = sorted(uniq, key=len, reverse=True)

        tags  = ["O"] * len(obs)
        taken = [False] * len(obs)

        # greedy non-overlapping BIO tagging
        for p in phrases:
            L = len(p)
            if L == 0 or L > len(obs):
                continue
            i = 0
            while i <= len(obs) - L:
                if (not any(taken[i:i+L])) and (obs[i:i+L] == p):
                    tags[i] = "B"
                    for j in range(1, L):
                        tags[i+j] = "I"
                    for j in range(L):
                        taken[i+j] = True
                    i += L
                else:
                    i += 1

        return tags

    #tokenize both our test and train data then run them through the BIO tagger
    tok_train, tok_test = [], []
    for obs, obj in zip(X_train, y_train):
        toks = spacy_tok(obs)
        tags = bio_from_object(toks, obj)
        tok_train.append(([str(t).lower().strip() for t in toks], [BIO2ID[t] for t in tags]))
    num_B = sum(tag == BIO2ID["B"] for _, yids in tok_train for tag in yids)
    num_I = sum(tag == BIO2ID["I"] for _, yids in tok_train for tag in yids)
    num_O = sum(tag == BIO2ID["O"] for _, yids in tok_train for tag in yids)

    print(f"Label counts -> B:{num_B} I:{num_I} O:{num_O}")
    assert num_B + num_I > 0, "BIO labeling produced no B/I tags — check tokenization & casing!"

    for obs, obj in zip(X_test, y_test):
        toks = spacy_tok(obs)
        tags = bio_from_object(toks, obj)
        tok_test.append(([str(t).lower().strip() for t in toks], [BIO2ID[t] for t in tags]))

    # get the counts of each word to keep track  unknowns and padding
    word_counts = Counter(t for toks,_ in tok_train for t in toks)
    word_counts = {w : c for w,c in word_counts.items() if c >= 2}
    itos = ["<pad>", "<unk>"] + sorted(list(word_counts.keys()))
    stoi = {w:i for i,w in enumerate(itos)}

    def numericalize(tokens): return [stoi.get(t, stoi["<unk>"]) for t in tokens]

    EMB_DIM = 16

    #intialize model
    class BiLSTMTagger(torch.nn.Module):
        def __init__(self, vocab_size, emb_dim, hidden_dim=8, num_labels=3):
            super().__init__()
            self.emb = torch.nn.Embedding(vocab_size, emb_dim, padding_idx=0)
            self.lstm = torch.nn.LSTM(emb_dim, hidden_dim, bidirectional=True, batch_first=True)
            self.fc = torch.nn.Linear(hidden_dim*2, num_labels)
        def forward(self, x, lengths):
            emb = self.emb(x)
            packed = torch.nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
            out,_ = self.lstm(packed)
            out,_ = torch.nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
            return self.fc(out)

    # helper to add padding, tokenize, do other data transforms for inference
    def collate(batch):
        xs, ys = zip(*batch)
        lens = torch.tensor([len(x) for x in xs], dtype=torch.long)
        maxlen = max(lens).item()
        xpad = torch.full((len(xs), maxlen), 0, dtype=torch.long)
        ypad = torch.full((len(xs), maxlen), BIO2ID["O"], dtype=torch.long)
        for i,(x,y) in enumerate(zip(xs, ys)):
            xpad[i,:len(x)] = torch.tensor(numericalize(x), dtype=torch.long)
            ypad[i,:len(y)] = torch.tensor(y, dtype=torch.long)
        return xpad, ypad, lens
    # hand made batch function used batch size 0 as that is what was given to me
    def batches(data, bs=32):
        for i in range(0, len(data), bs):
            yield data[i:i+bs]

    total_labels = 0
    label_freqs = Counter()
    for _, yids in tok_train:
        for lab in yids:
            label_freqs[lab] += 1
            total_labels += 1
    # Create weighting for B, I and O
    weights = []
    for label in range(3):
        freq = label_freqs.get(label, 1)
        weights.append(total_labels / (3 * freq))
    class_weights = torch.tensor(weights, dtype=torch.float32)

    model = BiLSTMTagger(len(itos), EMB_DIM )
    criterion = torch.nn.CrossEntropyLoss(weight= class_weights.to("cpu"))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    # create a way to save weghts so we don't have to continuously retrain
    # if os.path.exists('temp-weights.pt'):
    #     model.load_state_dict(torch.load('temp-weights.pt', weights_only=True))
    # else:
    print('> started training')
    epochs = 30
    model.train()
    for ep in range(1, epochs+1):
        total = 0.0
        pbar = tqdm(list(batches(tok_train, 32)))
        for batch in pbar:
            xpad, ypad, lens = collate(batch)
            logits = model(xpad, lens)
            B,T,C = logits.shape
            loss = criterion(logits.view(B*T, C), ypad.view(B*T))
            pbar.set_postfix(loss=f"{loss.item():.3f}")
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item()

    torch.save(model.state_dict(), 'temp-weights.pt')
    # a helper for model inference
    def decode_tags(tokens):
        model.eval()
        with torch.no_grad():
            xpad, _, lens = collate([(tokens, [BIO2ID["O"]]*len(tokens))])
            pred = model(xpad, lens).argmax(-1)[0].tolist()
        return [ID2BIO[i] for i in pred]

    # a helper for object extraction
    def extract_object_from_tags(tokens, tags):
        spans, cur = [], []
        for t, y in zip(tokens, tags):
            tok = str(t)
            if y == "B":
                if cur: spans.append(cur)
                cur=[t]
            elif y == "I":
                if cur: cur.append(t)
            else:
                if cur: spans.append(cur); cur=[]
        if cur: spans.append(cur)
        return " ".join(spans[0]) if spans else ""

    def final_inference_engine(obs):
        toks = spacy_tok(obs)
        tags = decode_tags(toks)
        return extract_object_from_tags(toks, tags)

    return final_inference_engine


print('Training object tagger...')
obj_policy = train_object_tagger([obs for obs, *_ in ml_data], [obj for _, _, obj in ml_data])

In [ ]:
class PipelineAgent:
        last_good_observation = None
        verbs_since_good = []

        def __init__(self, history_len: int = 6):
            self.history = deque(maxlen=history_len)
            self.last_obs_hash = None
            self.stale_steps = 0
            self.last_cmd = None
            self.MAX_STALE = 2

        # phrases TextWorld commonly uses for failed/invalid commands.
        FAIL_PATTERNS = [
            "you can't", "you cannot", "you don't", "you do not",
            "i don't understand", "i do not understand",
            "that's not a verb", "not a verb i recognise",
            "you aren't holding", "you are not holding",
            "there is no", "you see no", "nothing happens",
            "that doesn't make sense", "that does not make sense",
            "i only understood", "i only understood you as far as",
            "i didn't understand that sentence"
        ]

        MORE_PATTERNS = [
            "what do you want", "say which"
        ]

        VERBS_NO_OBJECT = {
            "look", "l", "inventory", "i", "wait", "help", "score", "save", "restore", "restart", "quit"
        }
        DIRECTIONS =  {
            "north", "south", "east", "west", "n", "s", "e", "w",
            "northeast", "northwest", "southeast", "southwest", "ne", "nw", "se", "sw",
            "up", "down", "u", "d", "enter", "exit"
        }

        VERBS = ['chop', 'cook', 'dice', 'drop', 'eat', 'examine', 'go', 'inventory', 'open',
            'prepare', 'slice', 'take']

        STOP_OBJS = {"you","i","me","that","this","sentence","any","such","thing"}

        NEED_OBJ_RE = re.compile(r"^what do you want to\s+([a-z]+)\??", re.I)



        def extract_exits(self, text: str):
            o = (text or "").lower()
            return [m.group(0) for m in re.finditer(
            r"\b(north|south|east|west|ne|nw|se|sw|up|down|enter|exit)\b", o)
                if m.group(0) in self.DIRECTIONS]

        last_good_observation = None
        verbs_since_good = []

        def _obs_sig(self, obs: str) -> tuple[int,int]:
            # cheap signature (len + hash) to detect “no real change”
            return (len(obs), hash(obs[:512]))

        def _diversify(self, verb: str, obj: str|None, cands: list[tuple[str,float]]) -> str:
            # avoid repeating exactly the same cmd; prefer a different object/verb
            tried = f"{verb} {obj}".strip()

            for cand_obj, _ in cands:
                alt_cmd = f"{verb} {cand_obj}".strip()
                if alt_cmd != tried:
                    return alt_cmd
            return "look"

        def looks_like_failure(self, obs):
            o = obs.lower()
            return any(pat in o for pat in self.FAIL_PATTERNS)

        def looks_like_more_needed(self, obs):
            o = obs.lower()
            return any(pat in o for pat in self.MORE_PATTERNS)

        def _update_history(self, observation: str, failure: bool=False, more_needed: bool=False):
            if not observation: return
            self.history.append(observation)

        def _context(self) -> str:
            # simple concat of recent “good” observations
            return "\n---\n".join(self.history)

        def _object_candidates(self, context: str, k: int = 4, for_verb: str | None =None) -> List[Tuple[str,float]]:


            cands: List[Tuple[str,float]] = []

            if for_verb == "go":
                for d in self.extract_exits(context):
                    cands.append((d, 0.9))

            primary = (obj_policy(context)).strip().lower()
            if primary:
                cands.append((primary, 1.0))

            toks = [t for t in spacy_tok(context)]
            seen, out = set(), []
            for t in toks:
                if len(t) > 2 and t not in seen and t not in self.STOP_OBJS:
                    seen.add(t); out.append(t)
                if len(out) >= k-1:
                    break
            cands.extend((s, 0.35) for s in out)
            # dedupe by best score
            best = {}
            for s, sc in cands:
                if s and (s not in best or sc > best[s]): best[s] = sc
            return sorted(best.items(), key=lambda p: (p[1], len(p[0])), reverse=True)[:k]

        def act(self, observation, info=None):

            is_failure = self.looks_like_failure(observation)
            needs_more = self.looks_like_more_needed(observation)
            is_good = not (is_failure or needs_more)

            ctx = self._context()

            if is_good:
                self._update_history(observation)
                self.last_good_observation = observation
                self.verbs_since_good = []
                ctx = ctx or observation
            else:
                if not ctx:
                    ctx = self.last_good_observation or observation

            # --- stale detection ---
            sig = self._obs_sig(observation)
            if self.last_obs_hash is not None and sig == self.last_obs_hash:
                self.stale_steps += 1
            else:
                self.stale_steps = 0
            self.last_obs_hash = sig

            # --- handle parser “need object” prompts without admissibles ---
            verb = None
            if needs_more:
                m = self.NEED_OBJ_RE.search(observation)
                if m:
                    verb = m.group(1).lower()

            # choose a verb if none extracted from prompt
            if not verb:
                verb = (verb_policy(ctx) or "").strip().lower()

            # diversify verb choice when stuck on failures/prompts
            if (is_failure or needs_more) and verb in self.verbs_since_good:
                unused = [v for v in self.VERBS if v not in self.verbs_since_good]
                if unused:
                    random.shuffle(unused)
                    verb = unused[0]
                else:
                    self.verbs_since_good = []
                    return "look"

            if not is_good:
                self.verbs_since_good.append(verb)

            # no-object verbs
            if verb in self.VERBS_NO_OBJECT:
                return verb

            # --- object selection (TEXT ONLY; no admissibles) ---
            obj = None

            # If prompt asked for an object, pick from recent good text
            if not obj and needs_more:
                try:
                    import spacy
                    nlp = getattr(self, "_nlp", None) or spacy.load("en_core_web_sm")
                    self._nlp = nlp
                    doc = nlp(ctx)
                    noun_chunks = [self._clean_object(ch.text) for ch in doc.noun_chunks]
                    noun_chunks = [nc for nc in noun_chunks if self._is_valid_object(nc)]
                    if noun_chunks:
                        random.shuffle(noun_chunks)
                        obj = noun_chunks[0]
                except Exception:
                    pass

            # General candidate mining from context/history (BIO + heuristics)
            if not obj:
                cands = self._object_candidates(ctx, k=6, for_verb=verb)
                if cands:
                    # use stale_steps as a simple diversification index
                    idx = min(self.stale_steps, len(cands) - 1)
                    obj = cands[idx][0] if isinstance(cands[0], tuple) else cands[idx]

            # Last fallback: BIO object policy directly
            if not obj:
                obj = self._clean_object(obj_policy(ctx))

            # Build command
            pred_cmd = f"{verb} {obj}".strip() if obj else verb

            # Extra diversification when stale (still without admissibles)
            if (self.stale_steps >= self.MAX_STALE or not is_good) and obj:
                cands = self._object_candidates(ctx, k=6, for_verb=verb)
                if len(cands) > 1:
                    # normalize list for _diversify (it can take tuples or strings)
                    pred_cmd = self._diversify(
                        verb,
                        obj,
                        cands  # can be [(obj,score), ...] per your current implementation
                    )

            self.last_cmd = pred_cmd
            return pred_cmd if obj else "look"


agent = PipelineAgent()
print('Example Run:')
files = list(os.listdir(gdir))
random.shuffle(files)
file = [f for f in files if f.endswith('.ulx')][0]
file = os.path.join(gdir, file)
game_id = textworld.gym.register_game(file)
env = textworld.gym.make(game_id)
obs, *_ = env.reset()
for i in range(100):
    print(f'> obs {i}:', obs)
    move = agent.act(obs)
    print(f'> move {i}:', move)
    obs, *_ = env.step(move)

In [ ]:
def make_env(ulx_path: str):
    gid = textworld.gym.register_game(ulx_path)
    return textworld.gym.make(gid)

def run_episode(env, agent, max_steps: int = 100) -> float:
    obs, *_ = env.reset()
    total = 0.0
    steps = 0
    success = False

    for _ in range(max_steps):
        outcome = (obs or "").lower()
        if "you lost" in outcome or "you have won" in outcome or "would you like to quit" in outcome:
            if "you have won" in outcome:
                success = True
            break
        move = agent.act(obs)
        obs, r, done, info = env.step(move)
        total += float(r)
        steps+=1
        if float(r) > 0 or info.get("won") or "you have won" in (obs or "").lower():
            success = True
        if done:
            break
    return total, steps, success

def evaluate(
    games_dir: str,
    agent,
    count = 0,
    total_steps = 0,
    success_count = 0,
    max_steps: int = 100,
    max_episodes: int | None = None,
    seed: int = 42,
    shuffle: bool = True
):
    random.seed(seed)
    ulx_files = [f for f in os.listdir(games_dir) if f.endswith(".ulx")]
    if shuffle:
        random.shuffle(ulx_files)
    if max_episodes is not None:
        ulx_files = ulx_files[:max_episodes]

    per_episode = []
    for f in ulx_files:
        ulx_path = os.path.join(games_dir, f)
        try:
            env = make_env(ulx_path)
        except Exception as e:
            print(f"[skip] {f}: {e}")
            continue
        try:
            ep_reward,steps, success = run_episode(env, agent, max_steps=max_steps)
            per_episode.append((f, ep_reward))
            total_steps += steps
            success_count += int(success)
            count+=1
        finally:
            try: env.close()
            except: pass

    total_reward = sum(r for _, r in per_episode)
    avg_steps = total_steps / count if count else 0.0
    success_rate = success_count / count
    return total_reward, per_episode, avg_steps, success_rate

total_reward,  per_episode, avg_steps, success_rate = evaluate(gdir, agent, max_steps=100, max_episodes=50)

print(f"Total reward over {len(per_episode)} episodes: {total_reward:.2f}")
print(f"Average step count: {avg_steps:.2f}")
print(f"Overall success rate: {success_rate * 100:.1f}%")
for fname, r in per_episode[:10]:  # print a few
    print(f"  {fname}: {r}")

## Part 2

In [ ]:
request_infos = textworld.EnvInfos(
    admissible_commands=True,
    inventory=True,
    description=True,
    objective=True,
    extras=["recipe"]
)

Dialogue

In [ ]:
DIALOGUE_QUESTIONS = {
    "admissible": "what are the admissible actions?",
    "inventory":  "what do i have?",
    "recipe":     "where is the recipe?",
    "objective":  "what is my objective?"
}

# Q -> A template functions (pull from request_infos)
def answer_admissible(info):
    acts = info.get("admissible_commands", []) or []
    return "The admissible actions are: " + "; ".join(acts) if acts else "I'm not sure."

def answer_inventory(info):
    inv = info.get("inventory")
    if inv and inv.strip():
        return "You have: " + inv
    return "I'm not sure."

def answer_recipe(info):
    # try extras['recipe'], fall back to anything that looks like it
    rec = None
    extras = info.get("extras") or {}
    if isinstance(extras, dict):
        rec = extras.get("recipe")
    if not rec:
        # sometimes objective hints mention the recipe
        rec = info.get("objective")
    return "The recipe is: " + str(rec) if rec else "I'm not sure."

def answer_objective(info):
    obj = info.get("objective")
    return "Your objective is: " + obj if obj else "I'm not sure."


DIALOGUE_ANSWERS = {
    "admissible": answer_admissible,
    "inventory":  answer_inventory,
    "recipe":     answer_recipe,
    "objective":  answer_objective
}

def is_dialogue(utterance: str) -> tuple[bool, str | None]:
    u = (utterance or "").strip().lower()
    # exact template matches (add fuzzy if you want)
    for key, q in DIALOGUE_QUESTIONS.items():
        if u == q:
            return True, key
    # optional: allow "ask admissible" shorthand
    if u.startswith("ask "):
        # ask admissible / ask inventory / ask recipe / ask objective
        tail = u[4:].strip()
        if tail in DIALOGUE_ANSWERS:
            return True, tail
    return False, None



NPC

In [ ]:
class NPC:
    def __init__(self, p_unsure=0.5):
        self.p_unsure = p_unsure
        self.cache = {}  # (act_key) -> response for this step

    def new_step(self):
        self.cache.clear()

    def respond(self, act_key: str, info: dict) -> str:
        # consistency within a step
        if act_key in self.cache:
            return self.cache[act_key]

        # coin-flip uncertainty
        if random.random() < self.p_unsure:
            self.cache[act_key] = "I'm not sure."
            return self.cache[act_key]

        # otherwise answer from request_infos
        fn = DIALOGUE_ANSWERS.get(act_key)
        response = fn(info) if fn else "I'm not sure."
        self.cache[act_key] = resp
        return resp

class EnvWithNPC:
    def __init__(self, env, npc):
        self.env = env
        self.npc = npc
        self.last_info = None  # store last info so NPC can read request_infos

    def reset(self):
        obs, info = self.env.reset()
        self.npc.new_step()      # start-of-episode consistency
        self.last_info = info
        return obs, info

    def step(self, action: str):
        # route
        is_dlg, key = is_dialogue(action)
        if is_dlg:
            # NPC answer; do NOT advance the env
            reply = self.npc.respond(key, self.last_info or {})

            obs = f"NPC: {reply}"
            reward = 0
            done = False
            info = self.last_info  # unchanged env info
            return obs, reward, done, info

        # real env step
        obs, reward, done, info = self.env.step(action)
        self.npc.new_step()   # a new game step => reset NPC cache
        self.last_info = info
        return obs, reward, done, info


Agent With NPC

In [ ]:
class PipelineAgentWithDialogue(PipelineAgent):

    def _maybe_ask(self, ctx: str) -> str | None:
        """
        Decide to ask NPC for help when stuck.
        Keep it simple: if stale for >= MAX_STALE, ask inventory (or objective).
        You can alternate which question to ask if you like.
        """
        if self.stale_steps >= self.MAX_STALE:
            # alternate between objective and inventory
            if (self.stale_steps % 2) == 0:
                return DIALOGUE_QUESTIONS["objective"]
            else:
                return DIALOGUE_QUESTIONS["inventory"]
        return None

    def act(self, observation, info=None):
        # Keep your existing act() logic — only add:
        # 1) include NPC observations into history
        if isinstance(observation, str) and observation.startswith("NPC:"):
            self._update_history(observation)

        # run your current act() to get a candidate command
        cmd = super().act(observation, info)

        # if we are stuck (i.e., super().act returned "look" repeatedly), ask NPC instead
        if cmd == "look":
            ask = self._maybe_ask(self._context() or observation or "")
            if ask:
                return ask

        return cmd

agent_w_npc = PipelineAgentWithDialogue()

Evaluate AGENT + NPC

In [ ]:
def run_eval(env_npc: EnvWithNPC, agent, episodes=10, max_steps=100):
    metrics = defaultdict(list)

    for ep in range(episodes):
        obs, info = env_npc.reset()
        total_reward, env_steps, dlg_steps = 0, 0, 0
        success = False

        for t in range(max_steps):
            action = agent.act(obs, info)
            is_dlg, _ = is_dialogue(action)

            obs, rew, done, info = env_npc.step(action)
            total_reward += rew
            if is_dlg: dlg_steps += 1
            else:      env_steps += 1

            if done:
                success = True
                break

        metrics["total_reward"].append(total_reward)
        metrics["env_steps"].append(env_steps)
        metrics["dlg_steps"].append(dlg_steps)
        metrics["success"].append(1 if success else 0)

    summary = {
        "avg_reward":  sum(metrics["total_reward"]) / len(metrics["total_reward"]),
        "avg_env_steps": sum(metrics["env_steps"]) / len(metrics["env_steps"]),
        "avg_dlg_steps": sum(metrics["dlg_steps"]) / len(metrics["dlg_steps"]),
        "success_rate": sum(metrics["success"]) / len(metrics["success"]),
    }
    return summary, metrics

In [ ]:
print('Example Run:')
files = list(os.listdir(gdir))
random.shuffle(files)
file = [f for f in files if f.endswith('.ulx')][0]
file = os.path.join(gdir, file)
game_id = textworld.gym.register_game(file,  request_infos=request_infos)
env = textworld.gym.make(game_id)
obs, *_ = env.reset()
for i in range(100):
    print(f'> obs {i}:', obs)
    move = agent.act(obs)
    print(f'> move {i}:', move)
    obs, *_ = env.step(move)



npc = NPC(p_unsure=0.5)                 # 50% "I'm not sure"
env_npc = EnvWithNPC(env, npc)


agent = PipelineAgentWithDialogue()


obs, info = env_npc.reset()
for t in range(20):
    a = agent.act(obs, info)
    print(f"[{t}] action:", a)
    obs, r, done, info = env_npc.step(a)
    print("obs:", obs.split("\n")[0][:120])
    if done: break

# Eval
summary, _ = run_eval(env_npc, agent, episodes=5, max_steps=100)
print("Summary:", summary)
